In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta

headers = {
    'Accept': 'application/json'
}

# Define the start and end dates for the year
start_date = datetime(2023, 1, 1)
end_date = datetime(2024, 1, 2)

# Function to get data for a given period
def get_data_for_period(start, end):
    from_ = start.strftime("%Y-%m-%dT%H:%MZ")
    to = end.strftime("%Y-%m-%dT%H:%MZ")
    r = requests.get(f"https://api.carbonintensity.org.uk/intensity/{from_}/{to}", headers=headers)
    return r.json()

# List to hold all data
all_data = []

# Loop through the year in 30-day increments
current_start = start_date
while current_start < end_date:
    current_end = current_start + timedelta(days=30)
    if current_end > end_date:
        current_end = end_date
    print(f"Fetching data from {current_start} to {current_end}")
    response = get_data_for_period(current_start, current_end)
    all_data.extend(response['data'])
    current_start = current_end

# Extract time and actual emissions
data = []
for entry in all_data:
    time = entry['from']
    actual_emissions = entry['intensity']['actual']
    data.append({'time': time, 'actual_emissions': actual_emissions})

# Create DataFrame
df1 = pd.DataFrame(data)

# Print DataFrame
df1


In [ ]:

df1.iloc[1:17443,:].to_csv("emissions_grid_intensity.csv", index=False)

grid = pd.read_csv("emissions_grid_intensity.csv", index_col=0, parse_dates=['time'])
grid = grid[~grid.index.duplicated()]
interp_grid = grid.resample("10min").interpolate(method='linear')
interp_grid = interp_grid.iloc[:-1, :]
interp_grid = interp_grid.reset_index()
interp_grid.columns = [" ","GRID CARBON INTENSITY (gCO2/kWh)"]
interp_grid.to_csv("grid_carbon_GB_10min_2023.csv", index=False)